# Sepsis Prediction Analysis

## Setup

### Packages

Upgrading some packages requires restarting the Colab runtime.
Check that they are installed and up to date to avoid having to restart later.

In [ ]:
# !pip install --upgrade ydata_profiling matplotlib tensorflow_addons "lightgbm<4.0.0"

### Download

In [ ]:
SETS = ("A", "B")
SET_DIR = "training_set{set_}"
URL = f"https://archive.physionet.org/users/shared/challenge-2019/{SET_DIR}.zip"

In [ ]:
import os
PATH = os.getcwd()
# try:
#     from google.colab import drive
#     mount_point = os.path.join(os.path.sep, "content", "drive")
#     drive.mount(mount_point)
#     PATH = os.path.join(mount_point, "My Drive", PATH)
# except ImportError:
#     pass
DATASET_DIR = os.path.join(PATH, "physionet")

In [ ]:
import shutil
set_path = os.path.join(DATASET_DIR, SET_DIR)
dirs_exist = all(os.path.exists(set_path.format(set_=set_)) for set_ in SETS)
# if not dirs_exist:
#     os.makedirs(DATASET_DIR, exist_ok=True)
#     for set_ in SETS:
#         os.system(f"wget -N -c {URL.format(set_=set_)}")
#         os.system(f"unzip -qq {SET_DIR.format(set_=set_)}.zip")
#     shutil.move("training", set_path.format(set_="A"))
#     shutil.move(SET_DIR.format(set_="B"), DATASET_DIR)

## Exploration

In this case, a [description of data](../data/description.md) is available:
read it before starting the analysis.

### Content

Define the path where the data is downloaded and list its contents.

In [ ]:
os.listdir(DATASET_DIR)

List the content of the a subdirectory.

In [ ]:
set_ = "A"
set_path = set_path.format(set_=set_)
import numpy as np
np.array(sorted(os.listdir(set_path)))

List all `.psv` files, check if the numbering is consecutive.
*Check for consistency*

In [ ]:
FILE_FORMAT = ".psv"
set_filenames = sorted(f for f in os.listdir(set_path) if f.endswith(FILE_FORMAT))
set_filenames[-1], int(len(set_filenames))

Numbering is _not_ consecutive then.

Check for patterns in missing patients.

In [ ]:
np.array([n for n in range(1, 20644) if f"p{n:06}.psv" not in set_filenames])

A single patient and a large chunk are missing.

Read one of the "pipe separated values" files,
and maybe repeat for different numbers.

In [ ]:
import pandas as pd
rng = np.random.default_rng(42) # open a random file

In [ ]:
filename = rng.choice(set_filenames)
file_df = pd.read_csv(os.path.join(set_path, filename), sep="|")
file_df

- 48 rows = 48 hours
- data has alot of NaN values
- Tabular data

This is expected for per-patient data.
Now the whole dataset can be read.

In [ ]:
from tqdm.auto import tqdm
patient_dfs = []
for set_ in SETS:
    set_path = os.path.join(DATASET_DIR, SET_DIR.format(set_=set_))
    set_filenames = sorted(f for f in os.listdir(set_path) if f.endswith(FILE_FORMAT))
    for filename in tqdm(set_filenames):
        patient = int(filename[1:-4])
        patient_path = os.path.join(set_path, filename)
        patient_df = pd.read_csv(patient_path, sep="|")
        patient_df["Patient"] = patient
        patient_df["Set"] = set_
        patient_dfs.append(patient_df)
df = pd.concat(patient_dfs).reset_index(drop=True)
df

- All data in a single dataframe.
- Ordered by Patient number
- 43 columns are too much to handle -> so ...
Divide the cells in groups as in the data description
to make the large number of different columns more manageable.

In [ ]:
vital_signs = df.columns[0:8].values
laboratory_values = df.columns[8:34].values
demographics = df.columns[34:40].values
outcome = df.columns[40]

### Patient statistics

Select cells that are expected to be constant for a given patient and check that they are.

In [ ]:
demographics

- Don't include length of stay (ICULOS), because it's not unique for each patient. It's rolling.

In [ ]:
# Patient will be analyzed and splits afterwards.
patient_df = df[["Patient", "Set", "Age", "Gender", "Unit1", "Unit2", "HospAdmTime"]].drop_duplicates() # one row for each patient
patient_df["Patient"].is_unique # check if patient number is unique

Check for missing values.

In [ ]:
patient_df.isna().any(axis=0)

Visualize the patient age distribution by sex and medical center.

In [ ]:
import plotly.express as px
px.histogram(patient_df, x="Age", color="Gender", facet_col="Set", barmode="overlay")

- Data are gathered differently in the both Sets A & B. In different resolution
   - eg.: Age is float in SetA and int in SetB
- In SetA they excluded the age in 100 (compare SetB)
- Distribution is consistent
  - Why is it important? -> If it is not consistent, model might learn different useful things (distribution shift).

No ages above 90 and accumulated ones around 100 are normal:
it is a standard protocol for de-identification
(the tail of the distribution always has only very few samples).

Note that the bin width chosen by `plotly` is a bit small:
one can see a lot of fluctuations due to too little statistics per bin.

Clearly age distributions are quite different.
Center "A" does not have either really young (<18) nor really old people (>89),
raising the interesting question of whether these patients were excluded from the study.
Also, the resolution of age for dataset "B" is lower than the one of dataset "A"
(try to zoom in any age range to see this).
Models trained on a dataset and evaluated on another
may have to perform inference out of distribution.

Now plot the unit type per gender and center;
fix the value of missing entries to -1 to visualize those too
(else they are ignored by `plotly`).

In [ ]:
px.histogram(patient_df.fillna(-1), x="Unit1", color="Gender", facet_col="Set", barmode="overlay")

In [ ]:
px.histogram(patient_df.fillna(-1), x="Unit2", color="Gender", facet_col="Set", barmode="overlay")

- Unit1 and Unit2 have 0/1
- Data collection is not the same, they have not the same entry

Distribution of the unit type is clearly different per center and gender,
but not critically so.
For missing values, it is unknown whether the type of unit was not recorded
or patients were assigned to a different type of ICU.
In the present dataset only MICU (Medical Intensive Care Unit)
and SICU (Surgical Intensive Care Unit) were recorded,
but depending on the hospital there are other types of ICU.

Moving on to hospital admission time...

In [ ]:
px.histogram(patient_df, x="HospAdmTime", color="Gender", facet_col="Set", barmode="overlay")

- hard to interpret plot.
- very wide range and a peak close to zero.
- if positive value vary a lot => use log

The distribution is so peaked that it is almost impossible to see anything.
In these cases use a logarithmic axis for the count!

In [ ]:
px.histogram(patient_df, x="HospAdmTime", color="Gender", facet_col="Set", barmode="overlay", log_y=True)

- Are the values always negative? No, there are some positiv values.

"HospAdmTime" is mostly negative, reaching down to very large numbers,
but there is a good minority of patients with positive values.

In [ ]:
print("There are", (patient_df["HospAdmTime"] > 0).sum(), "patients with positive hospital admission time")
print("and", (patient_df["HospAdmTime"] == 0).sum(), "patients with 0 hospital admission time")
patient_df["HospAdmTime"].describe()

From the data description, this variable represents the

> Hours between hospital admit and ICU admit

which would mean that patients are sometimes admitted to the ICU
_before_ being admitted to the hospital (negative values, up to 223 days!)
and sometimes _afterwards_ (up to a day).
Also, institution "B" does not have any positive entries, while "A" clearly does...

This may be a problem, _especially_ if through this variable a model is allowed to look into the far future
after the period of interest (e.g. the patient will not be dead in 2 weeks, so it's unlikely they have sepsis).


### Hourly features

Moving on from patients to single hourly measurements,
it is easy to get elementary statistics for all columns.
Proceed by group to make this more manageable.

**Task 1: analyze feature distributions and report your observations.**

The cells below include some checks for missing values and individual distributions per center.
Note that opening too many `plotly` figures with jupyter-lab may overload the browser,
so the default is limited.

In [ ]:
vital_signs_stats = df[vital_signs].describe().transpose()
vital_signs_stats["missing"] = 1 - vital_signs_stats["count"] / len(df)
vital_signs_stats

In [ ]:
for feature in vital_signs[:1]:
    display(px.histogram(df, x=feature, color="Set", barmode="overlay", log_y=True, nbins=100))

In [ ]:
for set_, set_df in df.groupby("Set"):
    print(set_)
    display(set_df[vital_signs].describe().transpose())

In [ ]:
laboratory_values_stats = df[laboratory_values].describe().transpose()
laboratory_values_stats["missing"] = 1 - laboratory_values_stats["count"] / len(df)
laboratory_values_stats

In [ ]:
for feature in laboratory_values[20:25]:
    display(px.histogram(df, x=feature, color="Set", barmode="overlay", log_y="True"))

In [ ]:
for set_, set_df in df.groupby("Set"):
    print(set_)
    display(set_df[laboratory_values].describe().transpose())

**Stop here, demographics were partly done already and the rest will be guided.**

In [ ]:
demographics_stats = df[demographics].describe().transpose()
demographics_stats["missing"] = 1 - demographics_stats["count"] / len(df)
demographics_stats

"Age", "Gender", "Unit1", "Unit2", and "HospAdmTime" were already analyzed.

"ICULOS" ranges from 1 to 336, which is expected from the data description
as 14 days * 24 hours / day gives 336 hours.
Is this really progressive patient-by-patient, starting from 1 until the end of ICU stay?

In [ ]:
counter = 0
for p, pdf in df.groupby("Patient"):
    if not (pdf["ICULOS"] == np.arange(len(pdf)) + 1).all():
        print("Patient", p, ": ICULOS", pdf["ICULOS"].values)
        counter += 1
        if counter > 10:
            break

It seems like this does not always start from 1.
Possibly this could be because in the first hours some data went missing...

Is "ICULOS" at least consecutive for all patients?

In [ ]:
counter = 0
for p, pdf in tqdm(df.groupby("Patient")):
    iculos = pdf["ICULOS"].sort_values().values
    if not ((iculos[1:] - iculos[:-1]) == 1).all():
        print("Patient", p, ": ICULOS", iculos)
        counter += 1
        if counter > 10:
            break

This is the case, which is reassuring about data quality.
What is the distribution of starting ICULOS?
And the one of ICULOS length?

In [ ]:
patient_df["ICULOSStart"] = None
patient_df["ICULOSEnd"] = None
patient_df["ICULOSLength"] = None
for p, pdf in tqdm(df.groupby("Patient")):
    patient_bools = patient_df["Patient"] == p
    patient_df.loc[patient_bools, "ICULOSStart"] = pdf["ICULOS"].min()
    patient_df.loc[patient_bools, "ICULOSEnd"] = pdf["ICULOS"].max()
    patient_df.loc[patient_bools, "ICULOSLength"] = len(pdf)

In [ ]:
px.histogram(patient_df, x="ICULOSStart", color="Gender", facet_col="Set", barmode="overlay", log_y=True)

There are 3 patients for which the series of available values starts after more than 10 days of ICU.
These are very likely difficult to handle, and removal is an option.
Otherwise, it is anyways not infrequent to find patients where recording starts more than 6 hours after ICU admission.

In [ ]:
px.histogram(patient_df, x="ICULOSEnd", color="Gender", facet_col="Set", barmode="overlay", log_y=True)

There is, for both sets, a drastic increase in discharges at 36 hours.
This may be well due to certain protocols which require observation in ICU for this amount of time.

There is also a drastic decrease in patients which remain in the ICU longer than 59 hours.
This might also be due to protocols which require patients to be transferred from the ICU (to palliative care?)
after a certain time lapse.

In [ ]:
px.histogram(patient_df, x="ICULOSLength", color="Gender", facet_col="Set", barmode="overlay", log_y=True)

### Outcome

Finally look at the outcome variable.

In [ ]:
df[outcome].describe()

Only 1.8% of the hourly windows are marked as "sepsis will be detected in 6 hours".
What about in terms of patients?

In [ ]:
patient_df[f"{outcome}Sum"] = [
    df[outcome][df["Patient"] == patient].sum()
    for patient in tqdm(patient_df["Patient"])
]

In [ ]:
patient_df[outcome] = patient_df[f"{outcome}Sum"] > 0

In [ ]:
px.histogram(patient_df, x=outcome, color="Gender", facet_col="Set", barmode="overlay")

In [ ]:
px.histogram(patient_df, x=f"{outcome}Sum", color="Gender", facet_col="Set", barmode="overlay")

In set B there are septic patients with data only up to a few hours immediately before sepsis.
For these patients, the actual time of sepsis is out of the recorded range!
Dropping them is an option.

In [ ]:
px.histogram(patient_df, x="Set", y=patient_df[outcome].astype(float), histfunc="avg", color="Gender", barmode="overlay")

The distributions across institutions and gender present significant but small differences.
The patients which develop sepsis range are around 8-9% for set "A" and 5-6% for set "B".

Look at the distribution of the onset of sepsis with length of stay,
both in terms of absolute numbers and as a fraction of records.

In [ ]:
px.histogram(df, x="ICULOS", y=outcome, color="Gender", facet_col="Set", barmode="overlay")

In [ ]:
px.histogram(df, x="ICULOS", y=outcome, histfunc="avg", color="Gender", facet_col="Set", barmode="overlay")

In [ ]:
px.histogram(df, x="ICULOS", color="Gender", facet_col="Set", barmode="overlay")

Note that the steep rise around 60 is probably due to
patients with suspicion of sepsis not being discharged.
Towards high values of ICULOS,
it is possible to distinguish individual septic patients,
given that the total number of samples is so low.


### Sepsis time

Reverse-engineer labels for the time at which sepsis really occurred, $t_\text{sepsis}$,
without the 6-hours advance.

In [ ]:
df["Sepsis"] = None
for p, pdf in tqdm(df.groupby("Patient")):
    iculos = pdf["ICULOS"]
    assert iculos.is_monotonic_increasing
    df_bools = df["Patient"] == p
    if pdf[outcome].iloc[0] == 0:
        # Patient developed sepsis after 6 hours or more of recorded data
        # Prepend 6 zeros to align with t_sepsis
        new_outcome = np.concatenate([np.zeros(6), pdf[outcome][:-6].values])
    elif iculos.iloc[0] <= 4:
        # According to the data description,
        # the patient developed sepsis between 4 and 6 hours of ICU
        new_outcome = pdf[outcome].values
        new_outcome[iculos <= 6] = -1  # Do not know
        new_outcome[iculos <= 4] = 0
    else:
        # If the recording starts with a higher ICULOS,
        # it is possible to have patients for which data starts already having sepsis,
        # which seems against the idea of sepsis prediction
        # (data which featured this at ICULOS 1 was discarded!)
        new_outcome = pdf[outcome].values
        new_outcome[:6] = -1  # Do not know
    df.loc[df_bools, "Sepsis"] = new_outcome

In [ ]:
df["Sepsis"] = df["Sepsis"].astype(int)
df["Sepsis"].value_counts()

Releasing data before preprocessing is a good idea.
The probability that something gets destroyed in cleanup is very high.
Rather release the code for preprocessing, and a preprocessed version of data if need be.

**Gold standards**

For every supervised Machine Learning problem,
there is the question of how to obtain accurate labels.
Procedures which are as accurate as possible for a given problem
take the name of *gold standards*.

According to the specific problem,
the gold standard may be for instance:

- the most accurate exam available (e.g. histopathology for dermatologic conditions);
- the opinion of one or more experts of the field;
- the consensus of a large number of people;
- ...

Whenever possible, the gold standard should be obtained
at least for the final evaluation data.
If the labels available for training are of lower quality,
sometimes one speaks of a *silver standard*.

**Learning from humans or nature**

Machine Learning algorithms can learn on data labelled by humans,
as was the case for the ECG dataset where experts determined the diagnosis.
However, in this case, it will be difficult in this case for an algorithm
to improve performance beyond the quality of the labels,
because the only additional signal it can learn is to correct mistakes and noise
in label assignment.
When the algorithm uses labels that were determined
using additional information that is not available from features,
instead, it may discover new patterns in data which were previously unknown to humans.
In this case, we say that the algorithm is *learning from nature*.

In the current case, according to the data description,
$t_\text{sepsis}$ is allowed to look into the future.
Namely, the label at a given time depends from laboratory results
and patient developments that are available only at a later time.
This is ok as long as it is our target variable,
i.e. not available to the model at the time of prediction.

In some sense, models learn from nature and not from humans:
not on the basis of expert opinions,
but using information not available to doctors at the time of prediction.

Cool!


### Automatic report

To complete data exploration, it is useful to generate a report automatically
and check if anything was missed.

In [ ]:
from ydata_profiling import ProfileReport
profile = ProfileReport(df)
profile.to_file("sepsis_report.html")

## Dataset splits

Example of evaluation for time series data:
number of arrivals in the emergency ward of hospitals.

- Case of data for 1 year from 300 different hospitals in a country.
- Case of data for 5 years from 3 hospitals in the same town.

**Task 2:**
**How does one perform evaluation in these cases?**
**What about the current analysis?**

In [ ]:
N_FOLDS = 3

In [ ]:
patient_df["Fold"] = -1
total = {"A": 0, "B": 0}
# Gender, Age, and Units should not be critical as there are enough samples
for (set_, label), df_ in patient_df.groupby(["Set", outcome]):
    shuffled_df = df_.sample(frac=1, random_state=df_["Patient"].min())
    for j in range(N_FOLDS):
        fold_patients = shuffled_df["Patient"][j::N_FOLDS]
        fold = (j + total[set_]) % N_FOLDS  # Rotate the number of the fold which gets residual elements
        patient_bools = patient_df["Patient"].isin(fold_patients)
        patient_df.loc[patient_bools, "Fold"] = fold
    total[set_] += len(shuffled_df)
for (set_, fold), df_ in patient_df.groupby(["Set", "Fold"]):
    print(f"Set {set_}, Fold {fold}: {len(df_)}")
    print(df_[outcome].value_counts())

In [ ]:
patient_df

In [ ]:
df["Fold"] = -1
for fold, patient_fold_df in patient_df.groupby("Fold"):
    patients = patient_fold_df["Patient"]
    fold_bools = df["Patient"].isin(patients)
    df.loc[fold_bools, "Fold"] = fold
df.value_counts(["Set", "Fold"]).sort_index()

For the present analysis, since time is limited,
we will reserve center "B" for testing,
and fold 2 of center "A" for validation.

## Preprocessing

Start by choosing the label column.
This could be the "SepsisLabel" as it was given,
which is positive from 6 hours before sepsis is diagnosed,
or the reverse-engineered "Sepsis",
which coincides with the time of the diagnosis.

Here the first is taken.

In [ ]:
label = outcome

### Feature drop

Check if there are features
which are completely missing in one of the sets,
and in that case drop them,
as they can only contribute negatively to cross-center comparison.

In [ ]:
for (set_, fold), df_ in df.groupby(["Set", "Fold"]):
    print(set_, fold, df.columns[df_.isna().all()])

In [ ]:
Xy = df.drop(columns="EtCO2")

In [ ]:
vital_signs = [f for f in vital_signs if f != "EtCO2"]

### Scaling

Create a `scikit-learn` compatible transformation
which implements column-wise robust scaling for multiple columns,
with the option to explicitly ignore certain columns
during the transformation

In [ ]:
columns_to_ignore = [
    "Patient", "Fold", "Set",
    "Sepsis", "SepsisLabel",
    "Gender", "Unit1", "Unit2",
    "Weights",
]

In [ ]:
from typing import Iterable, Optional

from sklearn.base import (
    BaseEstimator,
    TransformerMixin,
)

In [ ]:
class RobustScaler(TransformerMixin, BaseEstimator):

    def __init__(
        self,
        columns_to_ignore: Optional[Iterable[str]] = None,
        shift_quantile: float = 0.5,
        low_quantile: float = 0.25,
        high_quantile: float = 0.75,
    ):
        self.columns_to_ignore = []
        if columns_to_ignore is not None:
            self.columns_to_ignore = [c for c in columns_to_ignore]
        self.shift_quantile = shift_quantile
        self.low_quantile = low_quantile
        self.high_quantile = high_quantile
        self.xm = {}
        self.xr = {}

    def fit(self, X, y=None, verbose=False, **fit_params):
        for column in X.columns:
            if column in self.columns_to_ignore:
                continue
            xs = X[column]
            xm = xs.quantile(self.shift_quantile)
            xr = xs.quantile(self.high_quantile) - xs.quantile(self.low_quantile)
            if xr != 0:
                if verbose:
                    print(f"{column} will be shifted by {xm} and divided by {xr}")
            self.xm[column] = xm
            self.xr[column] = xr
        return self

    def transform(self, X, y=None):
        X = X.copy()
        for column in X.columns:
            if column in self.columns_to_ignore:
                continue
            if self.xr[column] != 0:
                X.loc[:, column] = (X[column] - self.xm[column]) / self.xr[column]
        return X

### Missing values

Build an object compatible with the `sklearn` API
which sets all NaNs below the minimum value allowed
for multiple columns,
and creates one-hot encoded columns identifying NaNs for linear models.

In [ ]:
class NanSetter(TransformerMixin, BaseEstimator):

    def __init__(
        self,
        columns_to_ignore: Optional[Iterable[str]] = None,
        na_column_name: str = "{column}Na",
    ):
        self.columns_to_ignore = []
        if columns_to_ignore is not None:
            self.columns_to_ignore = [c for c in columns_to_ignore]
        self.na_column_name = na_column_name
        self.nan_dict = {}

    def fit(self, X, y=None, verbose=False, **fit_params):
        for column in X.columns:
            if column in self.columns_to_ignore:
                continue
            xs = X[column]
            nas = xs.isna()
            if not nas.any():
                continue
            min_ = xs.min()
            max_ = xs.max()
            na_value = min_ - (max_ - min_)
            if verbose:
                print(f"Replacing NaNs in {column} with {na_value}")
            self.nan_dict[column] = na_value
        return self

    def transform(self, X, y=None):
        X = X.copy()
        for column in X.columns:
            if column in self.columns_to_ignore:
                continue
            nas = X[column].isna()
            if nas.any():
                X[self.na_column_name.format(column=column)] = nas.astype(float)
                X.loc[:, column] = X.fillna(self.nan_dict[column])
        return X

## Metrics

### Binary classification

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
)

### Clinical utility

Implement the clinical utility metric defined in [the evaluation file](evaluation.md).

In [ ]:
from math import floor

class Utility:
    def __init__(
        self,
        dt_early: float = -12.0,
        dt_optimal: float = -6.0,
        dt_late: float = 3.0,
        max_u_tp: float = 1.0,
        min_u_fn: float = -2.0,
        u_fp: float = -0.05,
        u_tn: float = 0.0,
        check_errors: bool = True,
    ):
        """Initialize all parameters needed for the computation
        of the clinical utility metric."""

        if dt_early >= dt_optimal:
            raise ValueError("The earliest beneficial time must be before the optimal time.")
        if dt_optimal >= dt_late:
            raise ValueError("The optimal time must be before the latest beneficial time.")
        # Define slopes and intercept points for utility functions of the form
        # u = m * t + b.
        self.m1 = max_u_tp / (dt_optimal - dt_early)
        self.b1 = -self.m1 * dt_early
        self.m2 = -max_u_tp / (dt_late - dt_optimal)
        self.b2 = -self.m2 * dt_late
        self.m3 = min_u_fn / (dt_late - dt_optimal)
        self.b3 = -self.m3 * dt_optimal
        # Store the rest
        self.dt_early = dt_early
        self.dt_optimal = dt_optimal
        self.dt_late = dt_late
        self.u_fp = u_fp
        self.u_tn = u_tn
        self.check_errors = check_errors

    def _check_errors(self, y_true, y_pred):
        """
        Check for inconsistencies when the metric is called
        with true and predicted labels.
        """
        if not self.check_errors:
            return
        if len(y_true) != len(y_pred):
            raise ValueError("`y_true` and `y_pred` have different length.")
        for y in y_true:
            if y not in (0, 1):
                raise ValueError("`y_true` must be 0 or 1.")
        for y in y_pred:
            if y not in (0, 1):
                raise ValueError("`y_pred` must be 0 or 1.")

    def u_pos(self, t, t_sepsis, y) -> float:
        """Compute prediction utility for patients who develop sepsis."""
        dt = t - t_sepsis
        if y:
            if dt <= self.dt_optimal:
                return max(self.m1 * dt + self.b1, self.u_fp)
            elif dt <= self.dt_late:
                return self.m2 * dt + self.b2
            else:
                return 0.0
        else:
            if dt <= self.dt_optimal:
                return 0.0
            elif dt <= self.dt_late:
                return self.m3 * dt + self.b3
            else:
                return 0.0

    def u_neg(self, t, y) -> float:
        """Compute prediction utility for patients who do not develop sepsis."""
        return self.u_fp if y else self.u_tn

    def u(self, t, t_sepsis, y):
        """Compute prediction utility for patients with or without sepsis."""
        if np.isfinite(t_sepsis):
            return self.u_pos(t, t_sepsis, y)
        else:
            return self.u_neg(t, y)

    def patient_u(
        self, y_true, y_pred,
    ) -> (float, float, float, float):
        """
        Compute the time of sepsis for a patientto get the total utility.

        The time of sepsis is set to infinity if patients never develop it.
        """
        self._check_errors(y_true=y_true, y_pred=y_pred)
        if np.any(y_true):
            t_sepsis = np.argmax(y_true) - self.dt_optimal
        else:
            t_sepsis = np.inf
        u = sum(self.u(t, t_sepsis, y) for t, y in enumerate(y_pred))
        y_best = np.zeros_like(y_true)
        if np.isfinite(t_sepsis):
            t_early = t_sepsis + self.dt_early
            y_best[max(0, int(floor(t_early))):] = 1
        best_u = sum(self.u(t, t_sepsis, y) for t, y in enumerate(y_best))
        zero_u = sum(self.u(t, t_sepsis, 0) for t, _ in enumerate(y_true))
        worst_u = sum(self.u(t, t_sepsis, 1-y) for t, y in enumerate(y_best))
        return u, best_u, zero_u, worst_u

    def call_patients(
        self, y_true, y_pred, patients
    ) -> float:
        """
        Call the utility metric given labels, predictions,
        and a column specifying patients.
        """
        u, best_u, zero_u, worst_u = 0.0, 0.0, 0.0, 0.0
        for patient in sorted(set(patients)):
            bools = patients == patient
            result = self.patient_u(y_true[bools], y_pred[bools])
            u += result[0]
            best_u += result[1]
            zero_u += result[2]
            worst_u += result[3]
        if u > zero_u:
            if best_u <= zero_u:
                raise ValueError(
                    f"The best utility function value {best_u} is not better "
                    f"than doing nothing {zero_u}."
                )
            return (u - zero_u) / (best_u - zero_u)
        else:
            if zero_u <= worst_u:
                raise ValueError(
                    f"The worst utility function value {worst_u} is better "
                    f"than doing nothing {zero_u}."
                )
            return (zero_u - u) / (zero_u - worst_u)

    def get_metric(self, patients):
        """
        Bind the metric to a column specifying patients,
        such that it can be called only with labels and predictions.
        """
        return lambda y_true, y_pred, weights=None: self.call_patients(y_true, y_pred, patients)

Test that the computation works.

In [ ]:
utility = Utility()
y_true = [0, 0, 1, 1, 1, 1, 1, 1]
y_pred = [0, 1, 1, 1, 1, 1, 1, 1]
utility.patient_u(y_true, y_pred)

Note: results on the full dataset are different
with respect to what one would obtain
if utility were computed _per patient_ and then averaged.

## Instantaneous detection

Start with a simple formulation of the problem:
predict if a patient is going to develop sepsis
from the present vital and laboratory values,
without considering their history.

In [ ]:
Xy_trainvalid = Xy[Xy["Set"] == "A"]
Xy_trainvalid = Xy_trainvalid[Xy_trainvalid[label] >= 0]
Xy_train = Xy_trainvalid[Xy_trainvalid["Fold"] != 2]
Xy_valid = Xy_trainvalid[Xy_trainvalid["Fold"] == 2]

In [ ]:
nan_setter = NanSetter()
scaler = RobustScaler(columns_to_ignore=columns_to_ignore)
Xy_train = nan_setter.fit_transform(scaler.fit_transform(Xy_train, verbose=True), verbose=True)
Xy_valid = nan_setter.transform(scaler.transform(Xy_valid))

### Evaluation function

In [ ]:
from typing import Protocol

class ScikitModel(Protocol):
    def fit(self, X, y, sample_weight=None): ...
    def predict(self, X): ...

In [ ]:
def get_scores(model: ScikitModel, X):
    try:
        return model.decision_function(X)
    except AttributeError:
        try:
            return model.predict_proba(X)[:, -1]
        except AttributeError:
            return model.predict_log_proba(X)[:, -1]

In [ ]:
from typing import Any, Callable, Dict, List, Optional
from imblearn.base import BaseSampler


def evaluate(
    Xy_train: pd.DataFrame,
    Xy_valid: pd.DataFrame,
    features: List[str],
    label: str,
    model: ScikitModel,
    metrics: Dict[str, Callable],
    threshold_metrics: Optional[Dict[str, Callable]] = None,
    sampler: Optional[BaseSampler] = None,
    sequential: bool = False,
) -> Dict[str, Any]:
    X_train = Xy_train[features]
    X_valid = Xy_valid[features]
    y_train = Xy_train[label]
    y_valid = Xy_valid[label]
    if sampler is not None:
        X_train, y_train = sampler.fit_resample(X_train, y_train)
    model.fit(X_train, y_train)
    result = {}
    y_pred = np.array(model.predict(X_valid))
    for k, v in metrics.items():
        result[k] = v(y_true=y_valid, y_pred=y_pred)
    if threshold_metrics is not None:
        scores = np.array(get_scores(model, X_valid))
        for k, v in threshold_metrics.items():
            result[k] = v(y_true=y_valid, y_score=scores)
    return result

### Dummy baselines

In [ ]:
from sklearn.dummy import DummyClassifier

In [ ]:
metrics = {
    "confusion_matrix": confusion_matrix,
    "accuracy": accuracy_score,
    "precision": precision_score,
    "recall": recall_score,
    "f1": f1_score,
    "matthews_corrcoef": matthews_corrcoef,
    "utility": utility.get_metric(patients=Xy_valid["Patient"]),
}
threshold_metrics = {
    "auc_roc": roc_auc_score,
    "auc_pr": average_precision_score,
}

In [ ]:
evaluate_kwargs = {
    "Xy_train": Xy_train,
    "Xy_valid": Xy_valid,
    "label": label,
    "metrics": metrics,
}

In [ ]:
baselines = ["most_frequent", "stratified", "uniform"]
strategy_results = []
for strategy in baselines:
    model = DummyClassifier(strategy=strategy)
    strategy_result = evaluate(features=[], model=model, **evaluate_kwargs)
    strategy_result["strategy"] = strategy
    strategy_results.append(strategy_result)
strategy_results = pd.DataFrame.from_records(strategy_results)
strategy_results

### Advanced baselines

In [ ]:
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.tree import ExtraTreeClassifier

In [ ]:
demographics = ["Age", "Gender", "Unit1", "Unit2"]
demographics_nan = [f"{fc}Na" for fc in demographics if f"{fc}Na" in Xy.columns]

In [ ]:
evaluate_kwargs["threshold_metrics"] = threshold_metrics

In [ ]:
evaluate(model=LogisticRegression(), features=(demographics + demographics_nan), **evaluate_kwargs)

In [ ]:
evaluate(model=LGBMClassifier(), features=demographics, **evaluate_kwargs)

In [ ]:
np.random.seed(42)
evaluate(model=ExtraTreeClassifier(), features=demographics, **evaluate_kwargs)

Models need to predict _for the same patient_
that the patient has no sepsis and sepsis at different times!
Tricky...

Let's add ICULOS and try again.

In [ ]:
demographics.append("ICULOS")

In [ ]:
evaluate(model=LogisticRegression(), features=(demographics + demographics_nan), **evaluate_kwargs)

In [ ]:
evaluate(model=LGBMClassifier(), features=demographics, **evaluate_kwargs)

In [ ]:
np.random.seed(42)
evaluate(model=ExtraTreeClassifier(), features=demographics, **evaluate_kwargs)

### Under- and over-sampling

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
undersampler = RandomUnderSampler(random_state=42)

In [ ]:
evaluate(model=LogisticRegression(), features=(demographics + demographics_nan), sampler=undersampler, **evaluate_kwargs)

In [ ]:
evaluate(model=LGBMClassifier(), features=demographics, sampler=undersampler, **evaluate_kwargs)

In [ ]:
np.random.seed(42)
evaluate(model=ExtraTreeClassifier(), features=demographics, sampler=undersampler, **evaluate_kwargs)

In [ ]:
from imblearn.over_sampling import RandomOverSampler
oversampler = RandomOverSampler()

**Task 3: repeat with oversampling**

*Skip the cells below until "Simple model" for now, come back later*

In [ ]:
X_train = Xy_train[demographics]
y_train = Xy_train[label]
X_valid = Xy_valid[demographics]
y_valid = Xy_valid[label]
X_train, y_train = oversampler.fit_resample(X_train, y_train)

In [ ]:
lgbm = LGBMClassifier(importance_type="split")
lgbm.fit(X_train, y_train)
print(lgbm.feature_importances_)

In [ ]:
lgbm = LGBMClassifier(importance_type="gain")
lgbm.fit(X_train, y_train)
print(lgbm.feature_importances_)

In [ ]:
import plotly.express as px

In [ ]:
ages = [X_valid["Age"].quantile(q) for q in np.linspace(0, 1, 11)]
genders = X_valid["Gender"].unique()
units = X_valid["Unit1"].unique()
iculoss = sorted(X_valid["ICULOS"].unique())
from itertools import product
grid_records = [x for x in product(ages, genders, units, units, iculoss)]
X_grid = pd.DataFrame(grid_records, columns=demographics)
y_grid = lgbm.predict(X_grid)

In [ ]:
px.line(X_grid, x="ICULOS", y=y_grid, facet_row="Unit1", facet_col="Unit2", color="Age", line_dash="Gender")

In [ ]:
y_pred_foo = X_valid["ICULOS"] > 1.25
{k: v(y_true=y_valid, y_pred=y_pred_foo) for k, v in metrics.items()}

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
pdp = PartialDependenceDisplay.from_estimator(
    estimator=lgbm,
    X=X_valid,
    features=demographics,
    percentiles=(0.01, 0.99),
    kind="both",
)
import matplotlib.pyplot as plt
plt.tight_layout()
plt.show()

### Simple model

In [ ]:
all_features = demographics + list(vital_signs) + list(laboratory_values)
all_features_nan = [f"{fc}Na" for fc in all_features if f"{fc}Na" in Xy.columns]

In [ ]:
evaluate(model=LogisticRegression(), features=(all_features + all_features_nan), sampler=oversampler, **evaluate_kwargs)

In [ ]:
evaluate(model=LGBMClassifier(), features=all_features, sampler=oversampler, **evaluate_kwargs)

In [ ]:
np.random.seed(42)
evaluate(model=ExtraTreeClassifier(), features=all_features, sampler=oversampler, **evaluate_kwargs)

### Previous values

In [ ]:
previous_df = df.drop(columns="EtCO2").copy()

In [ ]:
for p, pdf in tqdm(previous_df.groupby("Patient")):
    iculos = pdf["ICULOS"]
    assert iculos.is_monotonic_increasing
    previous_df.loc[pdf.index, :] = pdf.fillna(method="ffill")

In [ ]:
Xy_previous_trainvalid = previous_df[previous_df["Set"] == "A"]
Xy_previous_trainvalid = Xy_previous_trainvalid[Xy_previous_trainvalid[label] >= 0]
Xy_previous_train = Xy_previous_trainvalid[Xy_previous_trainvalid["Fold"] != 2]
Xy_previous_valid = Xy_previous_trainvalid[Xy_previous_trainvalid["Fold"] == 2]
previous_scaler = RobustScaler(columns_to_ignore=columns_to_ignore)
previous_nan_setter = NanSetter()
Xy_previous_train = previous_scaler.fit_transform(Xy_previous_train, verbose=True)
Xy_previous_valid = previous_scaler.transform(Xy_previous_valid)
Xy_previous_train = previous_nan_setter.fit_transform(Xy_previous_train, verbose=True)
Xy_previous_valid = previous_nan_setter.transform(Xy_previous_valid)

In [ ]:
evaluate_kwargs["Xy_train"] = Xy_previous_train
evaluate_kwargs["Xy_valid"] = Xy_previous_valid

In [ ]:
evaluate(model=LogisticRegression(), features=(all_features + all_features_nan), sampler=oversampler, **evaluate_kwargs)

In [ ]:
evaluate(model=LGBMClassifier(), features=all_features, sampler=oversampler, **evaluate_kwargs)

In [ ]:
np.random.seed(42)
evaluate(model=ExtraTreeClassifier(), features=all_features, sampler=oversampler, **evaluate_kwargs)

### Medical feature set

In [ ]:
medical_features = [
    "ICULOS",
    "HR", "Temp", "SBP", "MAP", "Resp",
    "Lactate", "Bilirubin_total", "WBC", "Platelets",# "PaCO2", "FiO2",
]
medical_features_nan = [f"{fc}_nan" for fc in medical_features if f"{fc}_nan" in Xy.columns]

In [ ]:
evaluate(model=LogisticRegression(), features=(medical_features + medical_features_nan), sampler=oversampler, **evaluate_kwargs)

In [ ]:
evaluate(model=LGBMClassifier(), features=medical_features, sampler=oversampler, **evaluate_kwargs)

In [ ]:
np.random.seed(42)
evaluate(model=ExtraTreeClassifier(), features=medical_features, sampler=oversampler, **evaluate_kwargs)

### Feature selection

In [ ]:
from typing import Iterable

def backward_elimination_dict(
    features: List[str],
    evaluate: Callable[Any, float],
    **kwargs,
) -> Dict[tuple, float]:
    n = len(features)
    score_dict = {}
    # Initialisation with all features
    key = tuple(sorted(features))
    score_dict[key] = evaluate(features=features, **kwargs)
    # Iteration
    current_features = [f for f in features]
    for _ in tqdm(range(n-1)):
        score_dict_n = {}
        for feature in current_features:
            new_features = [f for f in current_features if f != feature]
            score_dict_n[feature] = evaluate(features=new_features, **kwargs)
        print(score_dict_n)
        feature_to_eliminate, current_score = max(score_dict_n.items(), key=lambda x: x[1])
        current_features = [f for f in current_features if f != feature_to_eliminate]
        key = tuple(sorted(current_features))
        score_dict[key] = current_score
    return score_dict

In [ ]:
def evaluate_mcc(*args, **kwargs):
    return evaluate(*args, **kwargs)["matthews_corrcoef"]

In [ ]:
be_result = backward_elimination_dict(
    model=LGBMClassifier(),
    features=all_features,
    evaluate=evaluate_mcc,
    sampler=oversampler,
    **evaluate_kwargs,
)

In [ ]:
import pickle
with open("feature_selection_result.pkl", "wb") as file:
    pickle.dump(be_result, file)
be_result

In [ ]:
import pickle
with open("feature_selection_result.pkl", "rb") as file:
    be_result_loaded = pickle.load(file)
be_result_loaded

In [ ]:
plt.plot(be_result_loaded.values())
plt.xlabel("Features removed")
plt.ylabel("MCC")
plt.show()

In [ ]:
best_features = [k for k in be_result_loaded.keys()][26]
be_result_loaded[best_features]

In [ ]:
best_features = [f for f in best_features]
best_features

In [ ]:
be_model = LGBMClassifier()
evaluate(model=be_model, features=best_features, sampler=oversampler, **evaluate_kwargs)

### Other center

In [ ]:
Xy_previous_test = previous_df[previous_df["Set"] == "B"]
Xy_previous_test = Xy_previous_test[Xy_previous_test[label] >= 0]
Xy_previous_test = previous_scaler.transform(Xy_previous_test)
Xy_previous_test = previous_nan_setter.transform(Xy_previous_test)

In [ ]:
y_true_test = Xy_previous_test[label]
y_pred_test = be_model.predict(Xy_previous_test[best_features])
scores_test = get_scores(be_model, Xy_previous_test[best_features])

In [ ]:
patient_column_test = Xy_previous_test["Patient"]
metrics["utility"] = utility.get_metric(patient_column_test)

In [ ]:
result = {}
for k, v in metrics.items():
    result[k] = v(y_true=y_true_test, y_pred=y_pred_test)
for k, v in threshold_metrics.items():
    result[k] = v(y_true=y_true_test, y_score=scores_test)
result

In [ ]:
y_pred_baseline = Xy_previous_test["ICULOS"] > 1.25
result = {}
for k, v in metrics.items():
    result[k] = v(y_true=y_true_test, y_pred=y_pred_baseline)
result

In [ ]:
unique_patients = sorted(Xy_previous_test["Patient"].unique())
rng = np.random.default_rng(42)
bootstrap_result = []
for sample in range(10):
    sampled_patients = rng.choice(
        a=unique_patients,
        size=len(unique_patients),
        replace=True,
    )
    y_true_patient = []
    y_pred_patient = []
    patient_column = []
    for i, patient in tqdm(enumerate(sampled_patients)):
        y_true_patient.append(y_true_test[patient_column_test == patient])
        y_pred_patient.append(y_pred_test[patient_column_test == patient])
        patient_column.append(np.full_like(y_true_patient[i], fill_value=i))
    y_true_bootstrap = np.concatenate(y_true_patient)
    y_pred_bootstrap = np.concatenate(y_pred_patient)
    patient_column_bootstrap = np.concatenate(patient_column)
    metrics["utility"] = utility.get_metric(patient_column_bootstrap)
    for k, v in metrics.items():
        bootstrap_result.append([
            sample,
            k,
            v(y_true=y_true_bootstrap, y_pred=y_pred_bootstrap),
        ])
bootstrap_df = pd.DataFrame(bootstrap_result, columns=["sample", "metric", "value"])
bootstrap_df

In [ ]:
bootstrap_metric_df = bootstrap_df[bootstrap_df["metric"] != "confusion_matrix"]
px.box(bootstrap_metric_df, facet_col="metric", y="value")

## Sequential detection

### Preprocessing

In [ ]:
Xy_trainvalid = Xy[Xy["Set"] == "A"]
Xy_trainvalid = Xy_trainvalid[Xy_trainvalid[label] >= 0]
Xy_train = Xy_trainvalid[Xy_trainvalid["Fold"] != 2]
Xy_valid = Xy_trainvalid[Xy_trainvalid["Fold"] == 2]

In [ ]:
nan_setter = NanSetter()
scaler = RobustScaler(columns_to_ignore=columns_to_ignore)
Xy_train = nan_setter.fit_transform(scaler.fit_transform(Xy_train, verbose=True), verbose=True)
Xy_valid = nan_setter.transform(scaler.transform(Xy_valid))

### Generator

In [ ]:
Xy_train[f"{label}Float"] = Xy_train[label].astype(float)
Xy_valid[f"{label}Float"] = Xy_valid[label].astype(float)
train_class_weights = Xy_train[label].value_counts() ** -0.5
train_class_weights /= train_class_weights.sum()
valid_class_weights = Xy_valid[label].value_counts() ** 0.0
Xy_train["Weights"] = (train_class_weights)[Xy_train[label]].values
Xy_train["Weights"] *= (Xy_train[label] >= 0).astype(float)
Xy_valid["Weights"] = (valid_class_weights)[Xy_valid[label]].values
Xy_valid["Weights"] *= (Xy_valid[label] >= 0).astype(float)

In [ ]:
demographics = ["Age", "Gender", "Unit1", "Unit2", "ICULOS"]
feature_columns = demographics + list(vital_signs) + list(laboratory_values)
label_columns = [f"{label}Float"]
weight_columns = ["Weights"]

In [ ]:
def select_ts(Xy, patient):
    chunk = Xy.loc[Xy["Patient"] == patient].drop(columns=["Patient"])
    features = chunk[feature_columns]
    targets = chunk[label_columns]
    weights = chunk[weight_columns]
    return (features, targets, weights)

In [ ]:
def generate_ts(Xy):
    patients = sorted(Xy["Patient"].unique())
    for patient in patients:
        yield select_ts(Xy, patient)

In [ ]:
len_train = len(Xy_train["Patient"].unique())
def train_generator():
    return generate_ts(Xy_train)
len_valid = len(Xy_valid["Patient"].unique())
def valid_generator():
    return generate_ts(Xy_valid)

In [ ]:
train_generator_ = train_generator()
ts = next(train_generator_)
tuple(t.values.shape for t in ts)

### TF Dataset

In [ ]:
import tensorflow as tf

In [ ]:
shapes = (
    (None, len(feature_columns)),
    (None, len(label_columns)),
    (None, len(weight_columns)),
)
output_signature = tuple(tf.TensorSpec(shape=s, dtype=tf.float32) for s in shapes)
train_dataset = tf.data.Dataset.from_generator(
    generator=train_generator,
    output_signature=output_signature,
)
valid_dataset = tf.data.Dataset.from_generator(
    generator=valid_generator,
    output_signature=output_signature,
)

In [ ]:
BATCH_SIZE = 64
train_dataset = train_dataset.padded_batch(batch_size=BATCH_SIZE, padded_shapes=shapes)
valid_dataset = valid_dataset.padded_batch(batch_size=BATCH_SIZE, padded_shapes=shapes)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)
valid_dataset = valid_dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
iter_train_dataset = []
for i, xyw in enumerate(train_dataset.as_numpy_iterator()):
    iter_train_dataset.append(xyw)
    if i > 5:
        break
[[x.shape for x in data_point] for data_point in iter_train_dataset]

### Metrics

In [ ]:
import tensorflow_addons as tfa

class FromLogitsMixin:
    def __init__(self, from_logits=False, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.from_logits = from_logits

    def update_state(self, y_true, y_pred, sample_weight=None):
        if self.from_logits:
            y_pred = tf.nn.sigmoid(y_pred)
        return super().update_state(y_true, y_pred, sample_weight)


class TruePositives(FromLogitsMixin, tf.metrics.TruePositives):
    ...

class TrueNegatives(FromLogitsMixin, tf.metrics.TrueNegatives):
    ...

class FalsePositives(FromLogitsMixin, tf.metrics.FalsePositives):
    ...

class FalseNegatives(FromLogitsMixin, tf.metrics.FalseNegatives):
    ...

class BinaryAccuracy(FromLogitsMixin, tf.metrics.BinaryAccuracy):
    ...

class Precision(FromLogitsMixin, tf.metrics.Precision):
    ...

class Recall(FromLogitsMixin, tf.metrics.Recall):
    ...

class MatthewsCorrelationCoefficient(FromLogitsMixin, tfa.metrics.MatthewsCorrelationCoefficient):
    ...

class AUC(FromLogitsMixin, tf.metrics.AUC):
    ...

In [ ]:
tp = TruePositives(from_logits=True, name="tp")
tn = TrueNegatives(from_logits=True, name="tn")
fp = FalsePositives(from_logits=True, name="fp")
fn = FalseNegatives(from_logits=True, name="fn")

accuracy = BinaryAccuracy(from_logits=True, name="accuracy")
precision = Precision(from_logits=True, name="precision")
recall = Recall(from_logits=True, name="recall")

auc = AUC(curve='ROC', from_logits=True, name="auroc")
ap = AUC(curve='PR', from_logits=True, name="auprc")

metrics = [
    tp, tn, fp, fn,
    accuracy, precision, recall,
    auc, ap,
]

### Model

In [ ]:
probability_positive = Xy_train[label_columns].mean(axis=0).values
probability_negative = 1 - probability_positive
logit_positive = np.log(probability_positive/probability_negative)
logit_positive

In [ ]:
def _lower_triangular_mask(shape):
    """Creates a lower-triangular boolean mask over the last 2 dimensions."""
    row_index = tf.cumsum(tf.ones(shape=shape, dtype=tf.int32), axis=-2)
    col_index = tf.cumsum(tf.ones(shape=shape, dtype=tf.int32), axis=-1)
    return tf.greater_equal(row_index, col_index)

In [ ]:
def create_mask(x):
    batch_size, seq_len, dim = x.shape
    padding_mask = tf.cast(tf.math.reduce_any(tf.cast(x, tf.bool), axis=-1), tf.float32) # (batch_size, seq_len)
    scores = tf.matmul(x, x, transpose_b=True)
    scores_shape = tf.shape(scores)
    causal_mask_shape = tf.concat(
        [tf.ones_like(scores_shape[:-2]), scores_shape[-2:]], axis=0
    )
    causal_mask = tf.cast(_lower_triangular_mask(causal_mask_shape), x.dtype)
    return tf.minimum(
        padding_mask[:, tf.newaxis, :],
        causal_mask
    )

In [ ]:
example = tf.convert_to_tensor([
    [[5.0, 2.0], [7.0, 1.1], [9.5, -0.3]],
    [[1.5, 0.4], [1.1, 1.0], [0.0, 0.0]],
    [[5.1, -2.1], [0.0, 0.0], [0.0, 0.0]],
    [[-0.3, 1.1], [7.0, 1.0], [0.5, 6.3]],
])
example

In [ ]:
print(create_mask(example[:,:1,:]))
print(create_mask(example[:,:2,:]))
print(create_mask(example))

In [ ]:
N_FILTERS = 32
KERNEL_SIZE = 1
PRE_LAYERS = 3
POST_LAYERS = 1
NUM_HEADS = 1
DROPOUT = 0
input_ = tf.keras.layers.Input(shape=(None, len(feature_columns)), batch_size=BATCH_SIZE, dtype=tf.float32)
pre_convs = [
    tf.keras.layers.Conv1D(N_FILTERS, KERNEL_SIZE, padding="causal", activation="relu")
    for _ in range(PRE_LAYERS)
]
pre_norms = [
    tf.keras.layers.LayerNormalization(center=False, scale=False)
    for _ in range(PRE_LAYERS)
]
post_convs = [
    tf.keras.layers.Conv1D(N_FILTERS, KERNEL_SIZE, padding="causal", activation="relu")
    for _ in range(POST_LAYERS)
]
post_norms = [
    tf.keras.layers.LayerNormalization(center=False, scale=False)
    for _ in range(POST_LAYERS)
]
attention = tf.keras.layers.MultiHeadAttention(
    num_heads=NUM_HEADS, key_dim=N_FILTERS,
)
norm = tf.keras.layers.LayerNormalization(center=False, scale=False)
concat = tf.keras.layers.Concatenate()
final_concat = tf.keras.layers.Concatenate()
output = tf.keras.layers.Conv1D(1, 1)#, bias_initializer=tf.keras.initializers.Constant(logit_positive))

In [ ]:
result = input_
results = [input_]
for i in range(PRE_LAYERS):
    result = pre_norms[i](pre_convs[i](result))
    results.append(result)
result = concat(results)
attention_mask = create_mask(result)
result = attention(query=result, value=result, attention_mask=attention_mask) + result
result = norm(result)
results.append(result)
for i in range(POST_LAYERS):
    result = post_norms[i](post_convs[i](result))
    results.append(result)
result = output(final_concat(results))
model = tf.keras.Model(inputs=input_, outputs=result, name="Attention")
model.summary()

In [ ]:
data_point = iter_train_dataset[0]
inputs = data_point[0]
outputs = model(inputs)
outputs.shape

In [ ]:
# Check causality
steps = 42
x = outputs[:, :steps, :]
y = model(inputs[:, :steps, :])
tf.math.reduce_all((2 * (x - y) / (x + y)) < 1e-4)

In [ ]:
logits = model(tf.reshape(data_point[0], (1, -1, len(feature_columns))))
loss = tf.keras.losses.BinaryCrossentropy(from_logits=True)
loss(logits, tf.reshape(data_point[1], (1, -1, 1)), tf.reshape(data_point[2], (1, -1, 1)))

In [ ]:
tb_folder = "logs"
experiment_name = f"xe_bs{BATCH_SIZE}_f{N_FILTERS}_{model.name}_do{DROPOUT}_w05_2"
cp_folder = "checkpoints"
checkpoint_name = "checkpoint_{epoch:03d}.hdf5"

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=loss,
    metrics=metrics,
    weighted_metrics=[],
)

In [ ]:
model.evaluate(train_dataset)

### Training

In [ ]:
if not os.path.isdir(tb_folder):
    os.mkdir(tb_folder)
if not os.path.isdir(cp_folder):
    os.mkdir(cp_folder)
tensorboard_callback = tf.keras.callbacks.TensorBoard(
    log_dir=os.path.join(tb_folder, experiment_name)
)
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(cp_folder, experiment_name, checkpoint_name),
)
history = model.fit(
    x=train_dataset,
    validation_data=valid_dataset,
    epochs=20,
    callbacks=[tensorboard_callback, checkpoint_callback],
)